# Vuln2Action - CVE-only target_steps, expanded 80k, 1 epoch

Test whether expanding the CVE-only dataset beyond the 55,420 baseline rows improves
generation before trying longer schedules.

- target_steps, CVE-only, no CAPEC input
- expanded flow dataset: 80,477 rows, 64,388 train / 16,089 validation
- t5-base, LR 5e-5, warmup 0.1, 1 epoch
- batch 8/GPU, grad accum 4, 2x T4
- evaluate the final model on the same 1,000 baseline validation rows
- save checkpoint for a later full-validation run


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os
train_hits=sorted(glob.glob("/kaggle/input/**/flow_train.jsonl", recursive=True))
print("flow_train hits:", train_hits)
if not train_hits: raise FileNotFoundError("flow_train.jsonl not found")
FLOW_DIR=Path(train_hits[0]).parent
os.environ["FLOW_DIR"]=str(FLOW_DIR)
os.environ["TOKENIZERS_PARALLELISM"]="false"
print("FLOW_DIR:", FLOW_DIR)


In [ ]:
from pathlib import Path
import glob, os
val_hits=sorted(glob.glob("/kaggle/input/**/flow_validation.jsonl", recursive=True))
print("flow_validation hits:", val_hits)
common=[p for p in val_hits if "vuln2action-eval-common-1000" in p.replace("\\", "/")]
if not common:
    common=[p for p in val_hits if Path(p).parent != Path(os.environ["FLOW_DIR"])]
if not common: raise FileNotFoundError("common eval flow_validation.jsonl not found")
EVAL_DIR=Path(common[0]).parent
os.environ["EVAL_DIR"]=str(EVAL_DIR)
print("EVAL_DIR:", EVAL_DIR)


In [ ]:
import os, sys, subprocess
cmd=[
 sys.executable, "-m", "torch.distributed.run", "--nproc_per_node=2",
 "scripts/train_t5.py",
 "--dataset", os.environ["FLOW_DIR"],
 "--output-dir", "/kaggle/working/t5_stage1_expanded_target_steps_1ep",
 "--model-name", "t5-base",
 "--target-field", "target_steps",
 "--max-input-length", "256",
 "--max-target-length", "384",
 "--per-device-batch-size", "8",
 "--gradient-accumulation-steps", "4",
 "--learning-rate", "5e-5",
 "--warmup-ratio", "0.1",
 "--num-train-epochs", "1",
 "--save-steps", "1007",
 "--eval-steps", "1007",
 "--logging-steps", "100",
 "--save-total-limit", "1",
 "--no-load-best-model-at-end",
 "--fp16",
 "--gradient-checkpointing",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=os.environ.copy())


In [ ]:
from pathlib import Path
base=Path("/kaggle/working/t5_stage1_expanded_target_steps_1ep")
ckpts=sorted(base.glob("checkpoint-*"), key=lambda p:int(p.name.split("-")[-1]))
print("final exists:", (base/"final").exists())
print("checkpoints:", [p.name for p in ckpts])
if not (base/"final").exists() and not ckpts: raise FileNotFoundError("no model output")


In [ ]:
import os, sys, subprocess
from pathlib import Path
base=Path("/kaggle/working/t5_stage1_expanded_target_steps_1ep")
ckpt=base/"final"
if not ckpt.exists():
    ckpts=sorted(base.glob("checkpoint-*"), key=lambda p:int(p.name.split("-")[-1]))
    if not ckpts: raise FileNotFoundError("no checkpoint")
    ckpt=ckpts[-1]
out=Path("/kaggle/working/t5_stage1_expanded_target_steps_1ep_eval_common1000")
cmd=[
 sys.executable, "scripts/evaluate_t5.py",
 "--checkpoint", str(ckpt),
 "--dataset", os.environ["EVAL_DIR"],
 "--split", "validation",
 "--target-field", "target_steps",
 "--output-dir", str(out),
 "--max-input-length", "256",
 "--max-target-length", "384",
 "--batch-size", "4",
 "--num-beams", "4",
 "--limit", "1000",
 "--fp16",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=os.environ.copy())


In [ ]:
import json
from pathlib import Path
out=Path("/kaggle/working/t5_stage1_expanded_target_steps_1ep_eval_common1000")
m=json.loads((out/"metrics.json").read_text(encoding="utf-8"))
print(json.dumps(m, indent=2, ensure_ascii=False))
rows=[json.loads(l) for l in (out/"predictions.jsonl").open(encoding="utf-8")]
exact=sum(r["prediction"].strip()==r["reference"].strip() for r in rows)
print("exact:", exact, "/", len(rows), "unique predictions:", len({r["prediction"] for r in rows}))
for i,row in enumerate(rows[:3]):
    print(f"\n[{i}] {row['cve_id']}")
    print("REF:", row["reference"][:500])
    print("PRED:", row["prediction"][:500])
